In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat
import torch
import yaml
from NN_models import Model18, Model20, Model21, Model22, Model23, Model31, Model32
from MPCModelFuncs import differential_mpc_model, differential_mpc_model_no_load, differential_mpc_model_data_diff, differential_mpc_model_data_diff_and_load, planar_model_TV
from MPCModelFuncs import euler_step
import scipy.io as sio

# Data directory
dir = '../train_data/to_be_trained_T1_model_4_shuffled/'
model_id = 20
modelName = 'model_39_pre_as_bootcamp'
T = 1
h = 0.05
dynamic_model = 4
percentage_to_show = 0.045

x_test, y_test, states_test, states_test_next = [], [], [], []
# Load data
npz_files = [f for f in os.listdir(dir) if f.endswith(".npz")]
for file in npz_files:
    data = np.load(dir + file)
    x_test.append(data['x_test'])
    y_test.append(data['y_test'])
    states_test.append(data['states_test'])
    states_test_next.append(data['states_next_test'])
x_test = np.concatenate(x_test, axis=0)
y_test = np.concatenate(y_test, axis=0)
states_test = np.concatenate(states_test, axis=0)
states_test_next = np.concatenate(states_test_next, axis=0)

# Read yaml file
p = np.zeros(24)
with open('../../config/default.yaml') as stream:
    parameters = yaml.load(stream, Loader=yaml.SafeLoader)
    p[0] = parameters['model_params']['l_f']
    p[1] = parameters['model_params']['l_r']
    p[2] = parameters['model_params']['m']
    p[3] = parameters['model_params']['I_z']
    p[4] = parameters['model_params']['T_max_front']
    p[5] = parameters['model_params']['T_max_rear']
    p[6] = parameters['model_params']['T_brake_front']
    p[7] = parameters['model_params']['T_brake_rear']
    p[8] = parameters['model_params']['GR']
    p[9] = parameters['model_params']['eta_motor']
    p[10] = parameters['model_params']['r_wheel']
    p[11] = parameters['model_params']['g']
    p[12] = parameters['model_params']['C_roll']
    p[13] = parameters['model_params']['rho']
    p[14] = parameters['model_params']['C_d']
    p[15] = parameters['model_params']['C_l']
    p[16] = parameters['tyre_params']['B']
    p[17] = parameters['tyre_params']['C']
    p[18] = parameters['tyre_params']['D']
    p[19] = parameters['model_params']['downforce_front']
    p[20] = parameters['model_params']['downforce_rear']
    p[21] = parameters['model_params']['h_cog']
    p[22] = parameters['model_params']['track_width']
    p[23] = parameters['model_params']['diff_gain']

In [ ]:
if model_id == 18:
    model = Model18()
elif model_id == 20:
    model = Model20()
elif model_id == 21:
    model = Model21()
elif model_id == 22:
    model = Model22()
elif model_id == 23:
    model = Model23()
elif model_id == 31:
    model = Model31()
elif model_id == 32:
    model = Model32()
state_dict = torch.load('../saved_weights/' + modelName + '.pth')
model.load_state_dict(state_dict)
model.eval()

In [ ]:
states = states_test

control_actions = np.array([x_test[:,-1,0], x_test[:,-1,1], x_test[:,-2,0], x_test[:,-2,1]]).T

if dynamic_model == 0:
    dX = differential_mpc_model_no_load(states[:,-1,:], control_actions,p).T
elif dynamic_model == 1:
    dX = differential_mpc_model(states[:,-1,:], control_actions,p).T
elif dynamic_model == 2:
    dX = differential_mpc_model_data_diff(states[:,-1,:], control_actions,p).T
elif dynamic_model == 3:
    dX = differential_mpc_model_data_diff_and_load(states[:,-1,:], control_actions,p).T
elif dynamic_model == 4:
    dX = planar_model_TV(np.hstack((states[:,-1,:],states[:,-2,:])), control_actions,p,h).T
else:
    print('No valid dynamic model selected')
    exit()

# if dynamic_model == 0:
#     dX = differential_mpc_model_no_load(states, control_actions, p).T
# elif dynamic_model == 1:
#     dX = differential_mpc_model(states, control_actions, p).T
    
x_test = torch.from_numpy(x_test).float()
NN_correction = model(x_test)
states_next = states_test_next
states_next_model = euler_step(states[:,-1,:], dX, h)
states_next_NN = euler_step(states[:,-1,:], dX+NN_correction.detach().numpy(), h)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(50, 50), dpi=300)
size = len(states_next[:,0])

ax[0].plot(states_next[:int(percentage_to_show*size),0], label='True',linewidth=0.1)
ax[0].plot(states_next_model[:int(percentage_to_show*size),0], label='Model Predicted',linewidth=0.1, linestyle='--')
ax[0].plot(states_next_NN[:int(percentage_to_show*size),0], label='NN Predicted',linewidth=0.1, linestyle=':')
ax[0].set_xlabel('Timestep',fontsize=5)
ax[0].set_ylabel('Velocity x [m/s]',fontsize=5)
ax[0].legend(fontsize=2)

ax[1].plot(states_next[:int(percentage_to_show*size),1], label='True',linewidth=0.1)
ax[1].plot(states_next_model[:int(percentage_to_show*size),1], label='Model Predicted',linewidth=0.1, linestyle='--')
ax[1].plot(states_next_NN[:int(percentage_to_show*size),1], label='NN Predicted',linewidth=0.1, linestyle=':')
ax[1].set_xlabel('Timestep',fontsize=5)
ax[1].set_ylabel('Velocity y [m/s]',fontsize=5)
ax[1].legend(fontsize=2)

ax[2].plot(states_next[:int(percentage_to_show*size),2], label='True',linewidth=0.1)
ax[2].plot(states_next_model[:int(percentage_to_show*size),2], label='Model Predicted',linewidth=0.1, linestyle='--')
ax[2].plot(states_next_NN[:int(percentage_to_show*size),2], label='NN Predicted',linewidth=0.1, linestyle=':')
ax[2].set_xlabel('Timestep',fontsize=5)
ax[2].set_ylabel('Yaw Rate [rad/s]',fontsize=5)
ax[2].legend(fontsize=2)

plt.savefig("../saved_images/NN_comparison.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
RMSE_vx_model = np.sqrt(np.mean((states_next[:,0] - states_next_model[:,0])**2))
RMSE_vy_model = np.sqrt(np.mean((states_next[:,1] - states_next_model[:,1])**2))
RMSE_r_model = np.sqrt(np.mean((states_next[:,2]- states_next_model[:,2])**2))

RMSE_vx_NN = np.sqrt(np.mean((states_next[:,0] - states_next_NN[:,0])**2))
RMSE_vy_NN = np.sqrt(np.mean((states_next[:,1] - states_next_NN[:,1])**2))
RMSE_r_NN = np.sqrt(np.mean((states_next[:,2] - states_next_NN[:,2])**2))

max_vx_model = np.max(np.abs(states_next[:,0] - states_next_model[:,0]))
max_vy_model = np.max(np.abs(states_next[:,1] - states_next_model[:,1]))
max_r_model = np.max(np.abs(states_next[:,2] - states_next_model[:,2]))

max_vx_NN = np.max(np.abs(states_next[:,0] - states_next_NN[:,0]))
max_vy_NN = np.max(np.abs(states_next[:,1] - states_next_NN[:,1]))
max_r_NN = np.max(np.abs(states_next[:,2] - states_next_NN[:,2]))

std_vx_model = np.std(states_next[:,0] - states_next_model[:,0])
std_vy_model = np.std(states_next[:,1] - states_next_model[:,1])
std_r_model = np.std(states_next[:,2] - states_next_model[:,2])

std_vx_NN = np.std(states_next[:,0] - states_next_NN[:,0])
std_vy_NN = np.std(states_next[:,1] - states_next_NN[:,1])
std_r_NN = np.std(states_next[:,2] - states_next_NN[:,2])

print('RMSE of the true and predicted states:')
print("Model - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_model, RMSE_vy_model, RMSE_r_model))
print("NN    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_NN, RMSE_vy_NN, RMSE_r_NN))
print('Highest Error in:')
print("Model - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_model, max_vy_model, max_r_model))
print("NN    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_NN, max_vy_NN, max_r_NN))
print('Stadart Deviation:')
print("Model - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(std_vx_model, std_vy_model, std_r_model))
print("NN    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(std_vx_NN, std_vy_NN, std_r_NN))


In [ ]:
true = states_next[:int(percentage_to_show*size)]
model = states_next_model[:int(percentage_to_show*size)]
NN = states_next_NN[:int(percentage_to_show*size)]
data = {
    'true' : true,
    'model' : model,
    'NN' : NN,
}
# print(data['progress_vel_normal'])
# print(data['progress_vel_NN'])
sio.savemat('model_mismatch.mat', data)

In [ ]:
differential_mpc_model_no_load(np.expand_dims(np.array([1,1,1]),axis=0),np.expand_dims(np.array([1,0.39]),axis=0),p)

In [ ]:
test_data = loadmat('test_dynamics.mat')
vx = test_data['test']['vx']
vy = test_data['test']['vy']
r = test_data['test']['r']

throttles = test_data['test']['throttles']
steerings = test_data['test']['steerings']

vx_prev = test_data['test']['vx_prev']
vy_prev = test_data['test']['vy_prev']
r_prev = test_data['test']['r_prev']

throttles_prev = test_data['test']['throttles_prev']
steerings_prev = test_data['test']['steerings_prev']

vxN = test_data['test']['vxN'][0][0][0]
vyN = test_data['test']['vyN'][0][0][0]
rN = test_data['test']['rN'][0][0][0]

resizedZ = test_data['test']['resizedZ'][0][0]
zN_1 = test_data['test']['zN_1'][0][0]
zN = test_data['test']['zN'][0][0]

states = np.array([vx[0][0][0],vy[0][0][0],r[0][0][0]]).T
control_actions = np.array([throttles[0][0][0],steerings[0][0][0]]).T

size = (vx[0][0].shape[1])
# print(size)
# print(throttles_prev[0][0][0].shape)
x = np.zeros((size,2,5))

for i in range(size):
    x[i,0,:] = np.array([throttles_prev[0][0][0][i],steerings_prev[0][0][0][i],vx_prev[0][0][0][i],vy_prev[0][0][0][i],r_prev[0][0][0][i]])
    x[i,1,:] = np.array([throttles[0][0][0][i],steerings[0][0][0][i],vx[0][0][0][i],vy[0][0][0][i],r[0][0][0][i]])

X = torch.tensor(x, dtype=torch.float32)
dX = differential_mpc_model_no_load(states,control_actions,p).T
NN_correction = model(X)
print(X.shape, dX.shape, NN_correction.shape)
NN_correction = NN_correction.detach().numpy()
states_next_NN = euler_step(states, dX+NN_correction, h)
states_next_model = euler_step(states, dX, h)

print(states_next_model.shape, states_next_NN.shape, states.shape, np.concatenate([states[:,2], rN]).shape)

fig, ax = plt.subplots(1, 3)

ax[0].plot(states_next_NN[:,0],label='True')
ax[0].plot(np.concatenate([states[1:,0], vxN]),label='Dynamics',linestyle='--')
ax[0].plot(np.concatenate([resizedZ[-3,1:].T, zN_1[-3].T, zN[-3].T]), label='States',linestyle=':')
ax[0].plot(states_next_model[:,0],label='Model',linestyle='-.')
ax[0].legend()

ax[1].plot(states_next_NN[:,1],label='True')
ax[1].plot(np.concatenate([states[1:,1], vyN]),label='Dynamics',linestyle='--')
ax[1].plot(np.concatenate([resizedZ[-2,1:].T, zN_1[-2].T, zN[-2].T]), label='States',linestyle=':')
ax[1].plot(states_next_model[:,1],label='Model',linestyle='-.')
ax[1].legend()

ax[2].plot(states_next_NN[:,2],label='True')
ax[2].plot(np.concatenate([states[2:,2], rN]),label='Dynamics',linestyle='--')
ax[2].plot(np.concatenate([resizedZ[-1,2:].T, zN_1[-1].T, zN[-1].T]), label='States',linestyle=':')
ax[2].plot(states_next_model[:,2],label='Model',linestyle='-.')
ax[2].legend()

plt.show()

plt.plot(np.concatenate([resizedZ[-3:,:].T, zN_1[-3:].T, zN[-3:].T]))

In [ ]:
x = range(1)
print(x[1])